# Snippet from Math-Lyapunov-Stability.md


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

def rosenbrock(x, y):
    """Rosenbrock function (non-convex test case)."""
    return (1 - x)**2 + 100*(y - x**2)**2

def noisy_gradient(x, y, noise_std=0.1):
    """Gradient with additive noise."""
    grad = np.array([
        -2*(1 - x) - 400*x*(y - x**2),
        200*(y - x**2)
    ])
    noise = np.random.randn(2) * noise_std
    return grad + noise

# Initialize
np.random.seed(42)
x_traj = [np.array([-0.5, 2.0])]  # Start away from minimum (1, 1)
controller = LyapunovController(initial_radius=0.5, ema_alpha=0.3)
learning_rate = 0.001

for step in range(200):
    x = x_traj[-1]
    E_t = rosenbrock(x[0], x[1])  # Current energy
    
    # Propose step
    grad = noisy_gradient(x[0], x[1])
    x_proposal = x - learning_rate * grad
    E_tp1 = rosenbrock(x_proposal[0], x_proposal[1])
    
    # Gate decision
    r_new, accept, metrics = controller.lyapunov_gate(E_t, E_tp1)
    
    if accept:
        x_traj.append(x_proposal)
    else:
        x_traj.append(x)  # Stay put
        learning_rate *= 0.9  # Reduce step size on rejection

print(f"Final position: ({x_traj[-1][0]:.3f}, {x_traj[-1][1]:.3f})")
print(f"Final energy: {rosenbrock(x_traj[-1][0], x_traj[-1][1]):.3f}")
print(f"Acceptance rate: {controller.history.count('ACCEPT')/len(controller.history):.1%}")
